In [24]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import torch.optim as optim
import h5py

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
device

device(type='cuda')

In [4]:
hdf = h5py.File('../COVID-classification.hdf5', 'r', swmr=True)

In [35]:
len(hdf['folds/1/train/Normal'])

38003

In [36]:
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
from torchvision import transforms # Added to make the class runnable for testing

class HDF5Dataset(Dataset):
    """
    Final corrected PyTorch Dataset for HDF5 data using the structure:
    /folds/{fold_index}/{data_type}/{class_name}
    """
    CLASS_MAPPING = {
        'Normal': 0,
        'Pneumonia': 1,
        'COVID-19': 2
    }
    
    def __init__(self, hdf5_path, fold_index, data_type, transform=None):
        super().__init__()
        
        self.hdf5_path = hdf5_path
        self.transform = transform
        self.file = None # Will hold the lazy-loaded h5py file handle
        
        self.data_info = [] 
        self.total_length = 0

        if data_type == 'test':
            base_group_path = f'/folds/{data_type}'
        else:
            base_group_path = f'/folds/{fold_index}/{data_type}'

        # 1. Open file briefly to extract metadata (paths and lengths)
        with h5py.File(self.hdf5_path, 'r') as f:
            for class_name, class_label in self.CLASS_MAPPING.items():
                
                # --- CORRECT PATH: The image dataset IS the class name path ---
                dataset_path = f'{base_group_path}/{class_name}'
                # -------------------------------------------------------------
                
                if dataset_path not in f:
                     print(f"Warning: Dataset not found at '{dataset_path}'. Skipping.")
                     continue
                
                # CHECK: If the error persists, the object at f[dataset_path] IS A GROUP, not a dataset.
                # If it's a dataset, .shape[0] works fine.
                try:
                    dataset_len = f[dataset_path].shape[0]
                except AttributeError:
                    raise AttributeError(
                        f"Failed to get shape for path: {dataset_path}. "
                        f"The object is likely an h5py.Group, not an h5py.Dataset (image array). "
                        f"Check your HDF5 file structure."
                    )
                
                # Store only picklable data (path, label, length info)
                self.data_info.append({
                    'path': dataset_path,
                    'label': class_label,
                    'start_idx': self.total_length,
                    'end_idx': self.total_length + dataset_len
                })
                
                self.total_length += dataset_len
                
        if self.total_length == 0:
             raise ValueError(f"No data found for fold {fold_index} and type {data_type}.")

    @property
    def h5f(self):
        """Lazily opens the HDF5 file handle once per worker process."""
        if self.file is None:
            self.file = h5py.File(self.hdf5_path, 'r')
        return self.file

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        # ... (Index finding logic: uses self.data_info) ...
        current_data = None
        local_idx = -1
        for data in self.data_info:
            if idx >= data['start_idx'] and idx < data['end_idx']:
                current_data = data
                local_idx = idx - data['start_idx']
                break
        
        if current_data is None:
             raise IndexError(f"Index {idx} is out of bounds.")

        # 2. Load data from HDF5
        dataset_ref = self.h5f[current_data['path']] 
        image = dataset_ref[local_idx] 
        label_int = current_data['label']

        # 3. Preparation and Transforms
        image = image.astype(np.float32) 
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label_int, dtype=torch.long)

        return image, label

    def close(self):
        if self.file is not None:
            self.file.close()
            self.file = None

    def __del__(self):
        self.close()

In [37]:
# Standard AlexNet input normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.ToPILImage(),             # Convert NumPy array to PIL Image
    transforms.Resize((224, 224)),       # AlexNet input size
    # Add data augmentation here if needed (RandomCrop, RandomHorizontalFlip)
    transforms.ToTensor(),               
    transforms.Normalize(mean=mean, std=std) 
])

In [38]:
def get_alexnet_model(num_classes):
    """Loads AlexNet and modifies the final layer."""
    # Load a pre-trained AlexNet (highly recommended for transfer learning)
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    
    # Freeze the feature extraction layers if you want pure transfer learning
    # for param in model.features.parameters():
    #     param.requires_grad = False

    # Replace the final fully-connected layer (classifier[6])
    # The input features to this layer must remain the same
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, num_classes)
    
    # Move the model to the target device
    model.to(device)
    return model

# Constants
NUM_CLASSES = 3  # Normal (0), Pneumonia (1), COVID-19 (2)
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_FOLDS = 5 # As specified in your setup

# Loss Function and Optimizer are defined outside the loop, 
# but are *re-initialized* inside the fold loop for a fresh model start.
criterion = nn.CrossEntropyLoss()

In [39]:
# Standard AlexNet input normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# --- Enhanced Training Transforms ---
train_transforms = transforms.Compose([
    transforms.ToPILImage(),             # Converts NumPy array to PIL Image
    
    # 1. Augmentation
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # Crop and resize to 224x224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10), # Rotate by up to 10 degrees

    # 2. Final Conversion and Normalization
    transforms.ToTensor(),               
    transforms.Normalize(mean=mean, std=std) 
])

# --- Validation Transforms (Should remain deterministic) ---
val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224), # Center crop to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [40]:
def train_model(model, train_loader, val_loader, criterion, optimizer):
    """Inner training and validation loop for a single epoch."""
    
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        
        # Calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100 * correct_train / total_train
    
    # --- Validation Phase ---
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * inputs.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = running_val_loss / len(val_loader.dataset)
    val_acc = 100 * correct_val / total_val
    
    return epoch_loss, epoch_acc, val_loss, val_acc


# --- Main Cross-Validation Loop ---
all_fold_results = []
HDF5_FILE_PATH = '../COVID-classification.hdf5' # Ensure this path is correct

for fold in range(1,NUM_FOLDS):
    print(f"\n{'='*30}\n| STARTING FOLD {fold+1}/{NUM_FOLDS} |\n{'='*30}")

    # 1. DATA LOADERS for the Current Fold
    try:
        # Note: Assume train_transforms and val_transforms are defined
        train_dataset = HDF5Dataset(HDF5_FILE_PATH, fold, 'train', transform=train_transforms)
        val_dataset = HDF5Dataset(HDF5_FILE_PATH, fold, 'val', transform=val_transforms)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    except Exception as e:
        print(f"Skipping Fold {fold}: Failed to load data. Error: {e}")
        continue
    
    # 2. MODEL, OPTIMIZER (RE-INITIALIZED)
    model = get_alexnet_model(NUM_CLASSES)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    best_val_acc = 0.0
    fold_history = []
    
    # 3. EPOCH LOOP
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc, val_loss, val_acc = train_model(
            model, train_loader, val_loader, criterion, optimizer
        )
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
        
        fold_history.append({'epoch': epoch, 'val_acc': val_acc})

        # Save the best model for this fold
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'alexnet_fold_{fold}_best.pth')

    # Record the best result for this fold
    all_fold_results.append(best_val_acc)
    
    # Explicitly close HDF5 file handles
    train_dataset.close()
    val_dataset.close()


print(f"\n{'='*40}\n| CROSS-VALIDATION SUMMARY |\n{'='*40}")
print(f"Validation Accuracies per Fold: {all_fold_results}")
print(f"Average Validation Accuracy: {np.mean(all_fold_results):.2f}% +/- {np.std(all_fold_results):.2f}%")


| STARTING FOLD 2/6 |
Skipping Fold 1: Failed to load data. Error: Failed to get shape for path: /folds/1/train/Normal. The object is likely an h5py.Group, not an h5py.Dataset (image array). Check your HDF5 file structure.

| STARTING FOLD 3/6 |
Skipping Fold 2: Failed to load data. Error: Failed to get shape for path: /folds/2/train/Normal. The object is likely an h5py.Group, not an h5py.Dataset (image array). Check your HDF5 file structure.

| STARTING FOLD 4/6 |
Skipping Fold 3: Failed to load data. Error: Failed to get shape for path: /folds/3/train/Normal. The object is likely an h5py.Group, not an h5py.Dataset (image array). Check your HDF5 file structure.

| STARTING FOLD 5/6 |
Skipping Fold 4: Failed to load data. Error: Failed to get shape for path: /folds/4/train/Normal. The object is likely an h5py.Group, not an h5py.Dataset (image array). Check your HDF5 file structure.

| STARTING FOLD 6/6 |
Skipping Fold 5: Failed to load data. Error: Failed to get shape for path: /folds

In [42]:
import h5py
HDF5_FILE_PATH = '../COVID-classification.hdf5'

try:
    with h5py.File(HDF5_FILE_PATH, 'r') as f:
        # Check a problematic path, e.g., Fold 1, Validation, COVID-19
        test_path = '/folds/1/val/COVID-19' 
        
        if test_path in f:
            obj = f[test_path]
            print(f"Object type at {test_path}: {type(obj)}")
            print(f"Object shape: {obj.shape}")
        else:
            print(f"Path not found: {test_path}")
except Exception as e:
    print(f"Error during inspection: {e}")

Object type at /folds/1/val/COVID-19: <class 'h5py._hl.group.Group'>
Error during inspection: 'Group' object has no attribute 'shape'


In [ ]:
# Custom AlexNet

class AlexNet(nn.Module):
        def __init__(self, num_classes=10):
            super(AlexNet, self).__init__()
            self.layer1 = nn.Sequential(
                nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
                nn.BatchNorm2d(96),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size = 3, stride = 2))
            self.layer2 = nn.Sequential(
                nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
                nn.BatchNorm2d(256),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size = 3, stride = 2))
            self.layer3 = nn.Sequential(
                nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(384),
                nn.ReLU())
            self.layer4 = nn.Sequential(
                nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(384),
                nn.ReLU())
            self.layer5 = nn.Sequential(
                nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(256),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size = 3, stride = 2))
            self.fc = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(9216, 4096),
                nn.ReLU())
            self.fc1 = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(4096, 4096),
                nn.ReLU())
            self.fc2= nn.Sequential(
                nn.Linear(4096, num_classes))

        def forward(self, x):
            out = self.layer1(x)
            out = self.layer2(out)
            out = self.layer3(out)
            out = self.layer4(out)
            out = self.layer5(out)
            out = out.reshape(out.size(0), -1)
            out = self.fc(out)
            out = self.fc1(out)
            out = self.fc2(out)
            return out